# Preparations

## Packages

In [1]:
import os, pathlib, datetime, itertools, pickle, mne
import numpy as np
import pandas as pd

## Settings

In [ ]:
# maximal possible total number of non-color change snapshots (60 trials with 9 snapshots per trial)
# for some trials, 10 snapshots are available but only a maximum of 9 are counted per trial
max_nr_sf_epochs = 540 # 5 blocks * 15 trials * 9 snapshots

# maximal possible total number of non-color change snapshots (2 rs periods with a maximum of 150 snapshots each)
max_nr_rs_epochs = 300

# frequency bands for snapshot feature extraction
freq_bands = {"D": [1, 4],
              "T": [4, 8],
              "A": [8, 14],
              "Bl": [14, 23],
              "Bh": [23, 30]}


"# Dictionary of relevant column in demografic data (name: column position)\ndemografic_data_columns = {'subject': 0,\n                           'group': 1,\n                           'gender': 2,\n                           'age': 3,\n                           'handedness': 4,\n                           'aq': 5,\n                           'bdi': 6,\n                           'css': 7}"

Analysis settings

In [3]:
# define all condition levels for which grand averages should be calculated
grand_averages_of_interest = ["starfield", "rs_start", "vel0", "vel1", "vel2", "den1", "den2", "nc", "cc"]

# all condition combinations used for counting number of epochs per condition combination per subject
condition_combinations = ['rs',
                          'cc/vel0/den1', 'cc/vel0/den2', 'cc/vel1/den1', 'cc/vel1/den2', 'cc/vel2/den1', 'cc/vel2/den2',
                          'nc/vel0/den1', 'nc/vel0/den2', 'nc/vel1/den1', 'nc/vel1/den2', 'nc/vel2/den1', 'nc/vel2/den2']

## Directories

In [ ]:
# path to all study data
path_study_data = pathlib.Path("D:/EEGST data/")

# Folder containing sourcedata (behavioral data, demografic data, montages)
path_sourcedata = path_study_data / 'bids_dataset' / 'sourcedata'

# path to general preprocessing folder (input) 
path_data_preprocessing =  path_study_data / 'preprocessing'

# group level data (output)
path_data_grouplevel = path_study_data / 'group_level'
# Create folder if they do not yet exist
if not os.path.isdir(path_data_grouplevel):
    os.mkdir(path_data_grouplevel)

## Functions

### Helper Functions

#### *give_subject_filepath()*
Returns paths to subject files

In [5]:
def give_subject_filepath(subject):
    """
    Return the path to the preprocessing files for a given subject.

    Args:
        subject (int or str): Subject identifier.

    Returns:
        pathlib.Path: Path to the subject's preprocessing data directory.
    """

    # subject as three letter string (including leading zeros)
    subject = str(subject).zfill(3)

    # path to subject preprocessing files
    path_data_preprocessing_subj = path_data_preprocessing / subject

    return(path_data_preprocessing_subj)

#### *get_subject_list()*
Extracts and return a list of all subject names from bids folder 

In [ ]:
def get_subject_list(use_raw_demodata = False, exclude_subjects = True, single_group = "controls", selection = []):
    """
    Retrieve a list of subject IDs from demographic data based on filtering criteria.

    Args:
        use_raw_demodata (bool): If True, load raw demographic data. If False, use preprocessed data.
        exclude_subjects (bool): If True, exclude subjects not marked as included.
        single_group (str): Filter to a specific group. Use "controls" for group 1 or "patients" for group 2.
        selection (list[int]): Optional list of subject IDs to include. If provided, only these subjects are returned.

    Returns:
        list[int]: List of subject IDs that meet the specified criteria.
    """
    

    # Load either raw or preprocessed demografic data
    if use_raw_demodata:
        demo_data = pd.read_csv(path_sourcedata / 'demografic_data.csv', encoding='latin-1', sep=";")
    else:
        demo_data = pd.read_csv(pathlib.Path(path_data_grouplevel / 'demografic_availabledata.csv'), encoding='cp1252')

    # only keep subjects from specific group
    if single_group == "controls":
        demo_data = demo_data[demo_data.group == 1]
    elif single_group == "patients":
        demo_data = demo_data[demo_data.group == 2]

    # only keep subjects not marked as excluded
    if exclude_subjects:
        demo_data = demo_data[demo_data.included]

    # extract subject names
    subjects = [int(subject["subject"]) for _ , subject in demo_data.iterrows()]

    if selection:
        subjects = [subject for subject in subjects if subject in selection] 

    return(subjects)

In [ ]:
def get_subject_list(use_raw_demodata = False, exclude_subjects = True, single_group = "controls", selection = []):
    """
    Retrieve a list of subject IDs from demographic data based on filtering criteria.

    Args:
        use_raw_demodata (bool): If True, load raw demographic data. If False, use preprocessed data.
        exclude_subjects (bool): If True, exclude subjects not marked as included.
        single_group (str): Filter to a specific group. Use "controls" for group 1 or "patients" for group 2.
        selection (list[int]): Optional list of subject IDs to include. If provided, only these subjects are returned.

    Returns:
        list[int]: List of subject IDs that meet the specified criteria.
    """
    
    # Load either raw or preprocessed demografic data
    if use_raw_demodata:
        demo_data = pd.read_csv(path_sourcedata / 'demografic_data.csv', encoding='latin-1', sep=";")
    else:
        demo_data = pd.read_csv(pathlib.Path(path_data_grouplevel / 'demografic_availabledata.csv'), encoding='cp1252')

    # only keep subjects from specific group
    if single_group == "controls":
        demo_data = demo_data[demo_data.group == 1]
    elif single_group == "patients":
        demo_data = demo_data[demo_data.group == 2]

    # only keep subjects not marked as excluded
    if exclude_subjects:
        demo_data = demo_data[demo_data.included]

    # extract subject names
    subjects = [int(subject["subject"]) for _ , subject in demo_data.iterrows()]

    if selection:
        subjects = [subject for subject in subjects if subject in selection] 

    return(subjects)

### Analysis Functions

##### *create_demoandavailabledata_sheet()*
Create table including demografic data and information about available data / exclusion of participants

In [ ]:
def create_demoandavailabledata_sheet():
    """
    Create and save a CSV sheet with demographic data and available data information for each subject.

    This includes columns for exclusion, available snapshots, and number of epochs per condition.
    The output is saved as 'demografic_availabledata.csv' in the group-level data directory.
    """
    
    # load orginal data
    demo_data_original = pd.read_csv(path_sourcedata / 'demografic_data.csv', encoding='latin-1', sep = ";") 

    # create copy of data
    demo_data_short = demo_data_original.copy()

    # insert column for information about participant inclusion as column 2
    demo_data_short.insert(1, "included", True)

    # determine for each subject how many snapshots are available
    rs_snapshot_numbers = []
    nc_snapshot_numbers = []
    max_nc_snapshot_numbers = []
    nc_snapshots_available_ratios = []
    condition_combination_counts = []

    for _, row in demo_data_short.iterrows():

        # get path to subject preprocessing files
        path_data_preprocessing_subj = give_subject_filepath(int(row["subject"]))

        feature_and_ratings_file = path_data_preprocessing_subj / "16_features_and_behav.csv"
        if os.path.isfile(feature_and_ratings_file):

            # load data
            featurebehav_df = pd.read_csv(path_data_preprocessing_subj / "16_features_and_behav.csv")

            # compute number of available rs epochs (snapshots)
            rs_snapshots = sum(["rs_start" in condition for condition in featurebehav_df["condition"]])
            rs_snapshot_numbers.append(rs_snapshots)

            # load behavioral data to determine, how many snapshots each trial could maximally entail
            behav_df = pd.read_csv(path_data_preprocessing_subj / "15a_ct_behav.csv")

            snapshot_pertrial = []
            for i in range(1,76):
                snapshot_pertrial.append(sum([trial == i for trial in featurebehav_df["trial_nr"]]))

            max_snapshot_pertrial = [row["duration"] // 2 for _, row in behav_df.iterrows()]

            behav_df.insert(behav_df.shape[1], "max_snapshot_pertrial", max_snapshot_pertrial)
            behav_df.insert(behav_df.shape[1], "snapshot_pertrial", snapshot_pertrial)

            # save behav_df
            behav_df.to_csv(path_data_preprocessing_subj / "15a_ct_behav_freqs.csv")

            # create list of number of snapshots per trial
            snapshots_pertrial = [row["snapshot_pertrial"] for _, row in behav_df.iterrows() if row["condition_code"] >= 11 and row["condition_code"] <= 16]

            # sum up list but limit number of snapshots per trial to 9
            sum_snapshots_pertrial = sum([min([9, k]) for k in snapshots_pertrial])
            
            # compute ratio of maximally possible number of snapshots to available number of snapshots
            nc_snapshots_available_ratio = sum_snapshots_pertrial / max_nr_sf_epochs

            # append information to lists for all subjects 
            nc_snapshot_numbers.append(sum_snapshots_pertrial)
            max_nc_snapshot_numbers.append(max_nr_sf_epochs)
            nc_snapshots_available_ratios.append(nc_snapshots_available_ratio)

            # compute number of snapshots per condition combination
            condition_combination_count = []
            for combination in condition_combinations:
                condition_combination_count.append(len([epoch for epoch in featurebehav_df["condition"] if combination in epoch]))
            condition_combination_counts.append(condition_combination_count)
        else:
            # add NaN values
            rs_snapshot_numbers.append(np.nan)
            nc_snapshot_numbers.append(np.nan)
            max_nc_snapshot_numbers.append(np.nan)
            nc_snapshots_available_ratios.append(np.nan)
            condition_combination_counts.append(np.repeat(np.nan, len(condition_combinations)).tolist())

    # save data about available rs snapshots
    demo_data_short.insert(demo_data_short.shape[1], "available_rs_snapshots", rs_snapshot_numbers)
    demo_data_short.insert(demo_data_short.shape[1], "available_rs_snapshots_pro", [x / max_nr_rs_epochs for x in rs_snapshot_numbers])


    demo_data_short.insert(demo_data_short.shape[1], "max_nc_snapshot_numbers", max_nc_snapshot_numbers)
    demo_data_short.insert(demo_data_short.shape[1], "available_nc_snapshots", nc_snapshot_numbers)
    demo_data_short.insert(demo_data_short.shape[1], "available_nc_snapshots_pro", nc_snapshots_available_ratios)

    # transposes list of rows (one row per subject with one entry per condition combination) to get a list of columns (one list per condition combination with one entry per subject) 
    condition_combination_counts_columns = list(map(list, itertools.zip_longest(*condition_combination_counts, fillvalue=None)))

    # save data about number of epochs per condition combination
    for column_nr, column in enumerate(condition_combination_counts_columns):
        demo_data_short.insert(demo_data_short.shape[1], condition_combinations[column_nr], column)

    # exclude participants based on missing data
    demo_data_short['included'] = [subject['available_nc_snapshots_pro'] >= .75 for _, subject in demo_data_short.iterrows()]

    # save data
    demo_data_short.to_csv(path_data_grouplevel / 'demografic_availabledata.csv', index = False)

##### *create_grandaverages()*

In [9]:
def create_grandaverages(subjects):
    """
    Compute and save grand average epochs for each condition of interest across all subjects.

    Args:
        subjects (list): List of subject identifiers.

    Saves:
        Grand average .fif files for each condition in the 'grand_averages' directory.
    """

    # create dict of lists for storing subject wise averages
    group_averages= dict()
    for key in grand_averages_of_interest:
        group_averages[key]= list()
    grand_averages = group_averages.copy()

    for subject in subjects:
        print(f"Currently loading {subject}")

        # get path to subject preprocessing files
        path_data_preprocessing_subj = give_subject_filepath(subject)
        
        # load data
        subject_epochs = mne.read_epochs(path_data_preprocessing_subj / '12a_snapshot_epo.fif')

        # for each key compute averge epoch and save in group data frame
        for key in grand_averages_of_interest:
            group_averages[key].append(subject_epochs[key].average())

    # create a new folder for grand averages if it does not yet exist
    if not os.path.isdir(path_data_grouplevel / 'grand_averages'):
        os.mkdir(path_data_grouplevel / 'grand_averages')

    for key in grand_averages_of_interest:
        grand_averages[key] = mne.grand_average(group_averages[key])
        grand_averages[key].save(path_data_grouplevel / 'grand_averages' / f"grandaverage_{key}.fif", overwrite = True)


##### *create_groupepochpowers()*

In [ ]:
def create_groupepochpowers(subjects):
    """
    Compute and save group-level average FFT powers for each condition across all subjects.

    Args:
        subjects (list): List of subject identifiers.

    Saves:
        Pickle file 'group_powers.pkl' with group average powers for each condition.
    """

    # create dict of lists for storing subject wise averages
    group_averages= dict()
    for key in grand_averages_of_interest:
        group_averages[key] = list()

    for subject in subjects:

        print(f"Currently loading {subject}")
        # get path to subject preprocessing files
        path_data_preprocessing_subj = give_subject_filepath(subject)

        # load subject powers
        #subject_powers = np.load((pathlib.Path(path_data_preprocessing_subj) / '13a_snapshot_fftpowers.npy'), allow_pickle=True)
        subject_powers = np.load((pathlib.Path(path_data_preprocessing_subj) / '13a_snapshot_amps.npy'), allow_pickle=True)

        # load dataframe containing snapshot conditions
        features_and_ratings = pd.read_csv(path_data_preprocessing_subj / "16_features_and_behav.csv")
            
        # for each key compute averge epoch and save in group data frame
        for key in grand_averages_of_interest:

            # find all indexes of epochs for the current condition
            indexes_of_interest = [epoch["index"] for _ , epoch in features_and_ratings.iterrows() if key in epoch["condition"]]

            # average all epochs with the identified indices
            mean_power = subject_powers[indexes_of_interest].mean(axis = 0)

            # append to group list
            group_averages[key].append(mean_power)

    # save group mean powers
    with open(path_data_grouplevel / 'group_powers.pkl', 'wb') as f:
        pickle.dump(group_averages, f)

##### *create_group_crosscondition_correlations()*
Compute group averages (1 correlation matrix for the entire group) of subjectwise cross condition correlations of epoch features. Basis for representation similartiy analysis

In [11]:
def create_group_crosscondition_correlations(subjects):
    """
    Compute and save group-level cross-condition correlation matrices for all subjects.

    Args:
        subjects (list): List of subject identifiers.

    Saves:
        CSV files with average correlation matrices (overall and per frequency band) in the group-level directory.
    """

    group_correlation_matrices = pd.DataFrame()
    group_correlation_matrices_abs = pd.DataFrame()

    for subject in subjects:

        # get path to subject preprocessing files
        path_data_preprocessing_subj = give_subject_filepath(subject)

        correlation_matrix = pd.read_csv(path_data_preprocessing_subj / '14a_cond_corrmatrix.csv', index_col = 0)
        correlation_matrix_abs = pd.read_csv(path_data_preprocessing_subj / '14b_cond_corrmatrix_abs.csv', index_col = 0)

        group_correlation_matrices = pd.concat((group_correlation_matrices, correlation_matrix))
        group_correlation_matrices_abs = pd.concat((group_correlation_matrices_abs, correlation_matrix_abs))

        group_band_correlation_matrices = {}
        group_band_correlation_matrices_abs = {}
        for band in freq_bands.keys():
            group_band_correlation_matrices[band] = pd.DataFrame()
            group_band_correlation_matrices_abs[band] = pd.DataFrame()

        # load and aggregate band wise correlation matrices
        for band in freq_bands.keys():  
            band_correlation_matrix = pd.read_csv(path_data_preprocessing_subj / f'14c_cond_corrmatrix_{band}.csv', index_col = 0)
            group_band_correlation_matrices[band] = pd.concat((group_band_correlation_matrices[band], band_correlation_matrix))

            band_correlation_matrix_abs = pd.read_csv(path_data_preprocessing_subj / f'14d_cond_corrmatrix_{band}_abs.csv', index_col = 0)
            group_band_correlation_matrices_abs[band] = pd.concat((group_band_correlation_matrices_abs[band], band_correlation_matrix_abs))

    # compute average sample correlation matrix
    sample_correlation_matrix = group_correlation_matrices.groupby(level=0).mean()
    sample_correlation_matrix_abs = group_correlation_matrices_abs.groupby(level=0).mean()

    # save correlation matrix as csv
    sample_correlation_matrix.to_csv(path_data_grouplevel / 'sample_correlation_matrix.csv')
    sample_correlation_matrix_abs.to_csv(path_data_grouplevel / 'sample_correlation_matrix_abs.csv')

    # Compute averages for each band
    sample_correlation_matrix_band = {}
    sample_correlation_matrix_band_abs = {}

    for band in freq_bands.keys():
        # compute average sample correlation matrix
        sample_correlation_matrix_band[band] = group_band_correlation_matrices[band].groupby(level=0).mean()
        # save correlation matrix as csv
        sample_correlation_matrix_band[band].to_csv(path_data_grouplevel / f'sample_correlation_matrix_{band}.csv')

        # compute average sample correlation matrix
        sample_correlation_matrix_band_abs[band] = group_band_correlation_matrices_abs[band].groupby(level=0).mean()
        # save correlation matrix as csv
        sample_correlation_matrix_band_abs[band].to_csv(path_data_grouplevel / f'sample_correlation_matrix_abs_{band}.csv')


##### *merge_behavioralandfeaturedata()*
Concatenate behavioral data and combined feature/behavioral data for all subjects

In [ ]:
def merge_behavioralandfeaturedata(subjects):
    """
    Concatenate behavioral and feature/behavioral data across all subjects and save as group-level CSV files.

    Args:
        subjects (list): List of subject identifiers.

    Saves:
        - 'ct_behav.group.csv': Concatenated behavioral data for cognitive tasks.
        - 'rs_behav.group.csv': Concatenated behavioral data for resting state.
        - 'feature_and_ratings.group.csv': Concatenated feature and ratings data.
    """

    group_ct_behav_df = pd.DataFrame()
    group_rs_behav_df = pd.DataFrame()
    group_featurebehav_df = pd.DataFrame()

    for subject in subjects:

        print(f"Currently merging {subject}")

        # get path to subject preprocessing files
        path_data_preprocessing_subj = give_subject_filepath(subject)
        
        # load subject behavioral data
        ct_behav_df = pd.read_csv(path_data_preprocessing_subj / '15a_ct_behav.csv')
        rs_behav_df = pd.read_csv(path_data_preprocessing_subj / '15b_rs_behav.csv')
        featurebehav_df = pd.read_csv(path_data_preprocessing_subj / "16_features_and_behav.csv")

        # concatonate all data
        group_ct_behav_df = pd.concat([group_ct_behav_df, ct_behav_df], ignore_index=True)
        group_rs_behav_df = pd.concat([group_rs_behav_df, rs_behav_df], ignore_index=True)
        group_featurebehav_df = pd.concat([group_featurebehav_df, featurebehav_df], ignore_index=True)


    # turn subject-id into integer
    group_ct_behav_df['subject'] = group_ct_behav_df['subject'].astype(int)
    group_rs_behav_df['subject'] = group_rs_behav_df['subject'].astype(int)
    group_featurebehav_df['subject'] = group_featurebehav_df['subject'].astype(int) 

    # save data sets
    group_ct_behav_df.to_csv(path_data_grouplevel / 'ct_behav.group.csv', index = False)
    group_rs_behav_df.to_csv(path_data_grouplevel / 'rs_behav.group.csv', index = False)
    group_featurebehav_df.to_csv(path_data_grouplevel / 'feature_and_ratings.group.csv', index = False)

# Processing

In [ ]:
# ----- Settings for subjects -----
# Select subjects for processing / empty list means all subjects will be processed
subject_selection = []

# select subjects from specific group for processing
single_group = "controls" # "patients" # "controls" # "all" # False


# ----- Group processing -----

# Inform about start of data aggregation
print(f"======================================================================================================================")
print(f"======================================================================================================================")
print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Starting to Aggregate Data Across Group ===============")
print(f"======================================================================================================================")


# -----------------
# create one dataset contaning demografic information and information about available data per subject
print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Creating Demografic Data Sheet ===============")
create_demoandavailabledata_sheet()


# Retrieve list of all subjects from the specified group (use newly created table for demografic and available data with information about excluded subjects)
all_subjects = get_subject_list(use_raw_demodata = False, exclude_subjects = False, single_group = single_group, selection = subject_selection)


# -----------------
# create datasets of all epoch features including corresponding behavioral data (ratings and demo data)
print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Creating Group Snapshot Table (Features & Ratings) ===============")
merge_behavioralandfeaturedata(all_subjects)

print(f"----------------------------------------------------------------------------------------------------------------------")


# Retrieve list of included subjects from the specified group (use newly created table for demografic and available data with information about excluded subjects)
subjects = get_subject_list(use_raw_demodata = False, exclude_subjects = True, single_group = single_group, selection = subject_selection)


# -----------------
# Creates grand average epoch (average of all subjects and epochs)
print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Computing Grand Averages ===============")
create_grandaverages(subjects)

print(f"----------------------------------------------------------------------------------------------------------------------")




# -----------------
# computes the average correlation matrix across all subjectwise cross condition correlations of epoch features
print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Computing Crosscondition Correlations ===============")
create_group_crosscondition_correlations(subjects)

print(f"----------------------------------------------------------------------------------------------------------------------")



# Inform about end of group aggregation
print(f"======================================================================================================================")
print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Group Data Aggregation Complete ===============")
print(f"======================================================================================================================")
print(f"======================================================================================================================")

=============== 11:33:07 // Starting to Aggregate Data Across Group ===============
=============== 11:33:07 // Creating Demografic Data Sheet ===============
=============== 11:33:14 // Creating Group Snapshot Table (Features & Ratings) ===============
Currently merging 1
Currently merging 2
Currently merging 3
Currently merging 4
Currently merging 5
Currently merging 6
Currently merging 7
Currently merging 8
Currently merging 9
Currently merging 10
Currently merging 11
Currently merging 12
Currently merging 13
Currently merging 14
Currently merging 15
Currently merging 16
Currently merging 17
Currently merging 18
Currently merging 19
Currently merging 20
Currently merging 21
Currently merging 22
Currently merging 23
Currently merging 24
Currently merging 25
Currently merging 26
Currently merging 27
Currently merging 28
Currently merging 29
Currently merging 30
Currently merging 31
Currently merging 32
Currently merging 33
Currently merging 34
Currently merging 35
Currently merging 36

C:\Users\jordingma\AppData\Local\Temp\ipykernel_1564\2789440014.py:37: RuntimeWarning: This filename (D:\EEGST data\group_level\grand_averages\grandaverage_starfield.fif) does not conform to MNE naming conventions. All evoked files should end with -ave.fif, -ave.fif.gz, _ave.fif or _ave.fif.gz
  grand_averages[key].save(path_data_grouplevel / 'grand_averages' / f"grandaverage_{key}.fif", overwrite = True)
C:\Users\jordingma\AppData\Local\Temp\ipykernel_1564\2789440014.py:37: RuntimeWarning: This filename (D:\EEGST data\group_level\grand_averages\grandaverage_rs_start.fif) does not conform to MNE naming conventions. All evoked files should end with -ave.fif, -ave.fif.gz, _ave.fif or _ave.fif.gz
  grand_averages[key].save(path_data_grouplevel / 'grand_averages' / f"grandaverage_{key}.fif", overwrite = True)
C:\Users\jordingma\AppData\Local\Temp\ipykernel_1564\2789440014.py:37: RuntimeWarning: This filename (D:\EEGST data\group_level\grand_averages\grandaverage_vel0.fif) does not conform 

Overwriting existing file.
Identifying common channels ...
Overwriting existing file.
Identifying common channels ...
Overwriting existing file.
Identifying common channels ...
Overwriting existing file.
Identifying common channels ...


C:\Users\jordingma\AppData\Local\Temp\ipykernel_1564\2789440014.py:37: RuntimeWarning: This filename (D:\EEGST data\group_level\grand_averages\grandaverage_vel2.fif) does not conform to MNE naming conventions. All evoked files should end with -ave.fif, -ave.fif.gz, _ave.fif or _ave.fif.gz
  grand_averages[key].save(path_data_grouplevel / 'grand_averages' / f"grandaverage_{key}.fif", overwrite = True)
C:\Users\jordingma\AppData\Local\Temp\ipykernel_1564\2789440014.py:37: RuntimeWarning: This filename (D:\EEGST data\group_level\grand_averages\grandaverage_den1.fif) does not conform to MNE naming conventions. All evoked files should end with -ave.fif, -ave.fif.gz, _ave.fif or _ave.fif.gz
  grand_averages[key].save(path_data_grouplevel / 'grand_averages' / f"grandaverage_{key}.fif", overwrite = True)
C:\Users\jordingma\AppData\Local\Temp\ipykernel_1564\2789440014.py:37: RuntimeWarning: This filename (D:\EEGST data\group_level\grand_averages\grandaverage_den2.fif) does not conform to MNE na

Overwriting existing file.
----------------------------------------------------------------------------------------------------------------------
=============== 11:34:54 // Computing Crosscondition Correlations ===============


C:\Users\jordingma\AppData\Local\Temp\ipykernel_1564\2789440014.py:37: RuntimeWarning: This filename (D:\EEGST data\group_level\grand_averages\grandaverage_cc.fif) does not conform to MNE naming conventions. All evoked files should end with -ave.fif, -ave.fif.gz, _ave.fif or _ave.fif.gz
  grand_averages[key].save(path_data_grouplevel / 'grand_averages' / f"grandaverage_{key}.fif", overwrite = True)


----------------------------------------------------------------------------------------------------------------------
=============== 11:34:56 // Group Data Aggregation Complete ===============
